In [ ]:
import numpy as np
import json
from pathlib import Path

from gerrychain import Graph
from gerrychain.partition import recursive_tree_part

import networkx as nx
from networkx.readwrite import json_graph
import pandas as pd
import altair as alt
from scipy.stats import binom, norm

from binary_ensemble.stream import BenDecoder
from gerrytools.mgrp import RunContainer, RecomRunnerConfig, RecomRunInfo

In [ ]:
CACHE_DIR = Path("chain_runs_frcw")
CACHE_DIR.mkdir(exist_ok=True)
DEGREE_GRID = np.arange(0, 21)


def build_edges(rows, cols):
    edges = []
    for r in range(rows):
        for c in range(cols):
            node = r * cols + c
            if c < cols - 1:
                edges.append((node, node + 1))
            if r < rows - 1:
                edges.append((node, node + cols))
    edges = np.array(edges)
    return edges[:, 0], edges[:, 1]


def boundary_nodes(rows, cols):
    return np.array([r * cols + c for r in range(rows) for c in range(cols)
                     if r in (0, rows - 1) or c in (0, cols - 1)])


def grid_to_json(m, n):
    nx_graph = nx.grid_2d_graph(m, n)
    nx.set_node_attributes(nx_graph, 1, "population")
    relabel = {(r, c): r * n + c for r in range(m) for c in range(n)}
    g_int = nx.relabel_nodes(nx_graph, relabel, copy=True)

    G = Graph.from_networkx(nx_graph.copy())
    init = recursive_tree_part(G, parts=range(m), pop_target=m, pop_col="population", epsilon=0.0)
    init_relabel = {relabel[k]: v + 1 for k, v in init.items()}
    nx.set_node_attributes(g_int, init_relabel, "assignment")

    path = CACHE_DIR / f"graph_{m}x{n}.json"
    with open(path, "w") as f:
        json.dump(json_graph.adjacency_data(g_int), f)
    return path


def run_seed_frcw(container, config, seed, m, total_steps):
    run_info = RecomRunInfo(
        pop_col="population",
        assignment_col="assignment",
        variant="R",
        balance_ub=1,
        n_steps=total_steps,
        pop_tol=0.0,
        n_threads=1,
        batch_size=1,
        writer="ben",
        rng_seed=seed,
    )
    container.run(run_info)

    output_path = Path(config.output_folder) / f"RecomR_assignment_{seed}_{total_steps}.jsonl.ben"
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(
            f"frcw produced no output for seed {seed} (m={m}, steps={total_steps}) - "
            f"check the log at {config.log_file(run_info)}"
        )
    return output_path


def run_all_seeds_frcw(m, n, total_steps, seeds):
    json_path = grid_to_json(m, n)
    config = RecomRunnerConfig(
        json_file_path=str(json_path),
        output_folder=str(CACHE_DIR / "output"),
        log_folder=str(CACHE_DIR / "logs"),
    )
    paths = {}
    with RunContainer(config) as rc:
        for seed in seeds:
            paths[seed] = run_seed_frcw(rc, config, seed, m, total_steps)
    return paths


def plan_degrees(a, U, V, bnodes):
    cut = a[U] != a[V]
    pairs = np.unique(np.sort(np.stack([a[U][cut], a[V][cut]], axis=1), axis=1), axis=0)
    districts = np.unique(a)
    degrees = np.bincount(pairs.ravel(), minlength=districts.max() + 1)[districts]
    return degrees, np.isin(districts, a[bnodes])


def decode_degrees(path, U, V, bnodes):
    degs, flags = [], []
    for a in BenDecoder(path):
        d, b = plan_degrees(np.asarray(a), U, V, bnodes)
        degs.append(d)
        flags.append(b)
    return np.concatenate(degs), np.concatenate(flags)


def degree_data(n, total_steps, seed):
    path = CACHE_DIR / f"revrecom_degrees_{n}x{n}_seed_{seed}_{total_steps}.npz"
    if path.exists():
        data = np.load(path)
        return data['degrees'], data['boundary']
    ben_path = run_all_seeds_frcw(n, n, total_steps, [seed])[seed]
    U, V = build_edges(n, n)
    degrees, boundary = decode_degrees(ben_path, U, V, boundary_nodes(n, n))
    np.savez(path, degrees=degrees, boundary=boundary)
    return degrees, boundary


def interior_pmf(degs, flags):
    interior = degs[~flags].astype(int)
    counts = np.bincount(interior, minlength=len(DEGREE_GRID))[:len(DEGREE_GRID)]
    return counts / counts.sum()


def discretized_binomial_pmf(N, p, grid=DEGREE_GRID):
    pmf = binom.pmf(grid, N, p)
    return pmf / pmf.sum()


def discretized_normal_pmf(mu, sigma, grid=DEGREE_GRID):
    pmf = norm.cdf(grid + 0.5, loc=mu, scale=sigma) - norm.cdf(grid - 0.5, loc=mu, scale=sigma)
    return pmf / pmf.sum()


def dict_to_pmf(prob_dict, grid=DEGREE_GRID):
    pmf = np.array([prob_dict.get(int(d), 0.0) for d in grid])
    return pmf / pmf.sum()


def panel_df(sources, sizes):
    rows = []
    for n in sizes:
        for d in range(1, 13):
            rows.append({'n': n, 'Degree': d, 'Source': 'Empirical p_n(d | I)', 'Probability': float(interior_pmfs[n][d])})
            for name, pmf in sources.items():
                rows.append({'n': n, 'Degree': d, 'Source': name, 'Probability': float(pmf[d])})
    return pd.DataFrame(rows)


def make_panel_chart(df, source_order, title, sizes):
    return alt.Chart(df).mark_line(point=True).encode(
        x=alt.X('Degree:O', title='Degree'),
        y=alt.Y('Probability:Q', title='Probability'),
        color=alt.Color('Source:N', sort=source_order),
        column=alt.Column('n:O', title='Grid size n', sort=sizes),
    ).properties(width=170, height=200, title=title)

In [ ]:
seed = 2024
sizes = list(range(4, 21))
total_steps = 10_000
PANEL_SIZES = [10, 15, 20]
# sizes = list(range(4, 76))
# total_steps = 100_000
# PANEL_SIZES = [20, 30, 50, 75]

In [ ]:
degree_arrs, boundary_arrs = {}, {}
for n in sizes:
    degree_arrs[n], boundary_arrs[n] = degree_data(n, total_steps, seed)

interior_pmfs = {n: interior_pmf(degree_arrs[n], boundary_arrs[n])
                 for n in sizes if (~boundary_arrs[n]).any()}

In [ ]:
POISSON_VORONOI_SIDEDNESS = {3: .011, 4: .107, 5: .259, 6: .295, 7: .199, 8: .090, 9: .030, 10: .007}

REFERENCE_BINOMIAL_LABEL = 'Binomial(7, 0.857)'
REFERENCE_NORMAL_LABEL = 'Discrete N(6,1)'
POISSON_VORONOI_LABEL = 'Poisson-Voronoi sidedness'

candidates = {
    REFERENCE_BINOMIAL_LABEL: discretized_binomial_pmf(7, 6 / 7),
    REFERENCE_NORMAL_LABEL: discretized_normal_pmf(6.0, 1.0),
    POISSON_VORONOI_LABEL: dict_to_pmf(POISSON_VORONOI_SIDEDNESS),
}

In [ ]:
tv_df = pd.DataFrame([
    {'n': n, 'Candidate': name, 'TV_distance': 0.5 * np.abs(pmf_emp - pmf).sum()}
    for n, pmf_emp in sorted(interior_pmfs.items())
    for name, pmf in candidates.items()
])

tv_chart = alt.Chart(tv_df).mark_line(point=True, size=2).encode(
    x=alt.X('n:Q', title='Grid size n'),
    y=alt.Y('TV_distance:Q', title='TV distance from p_n(d | Interior)'),
    color=alt.Color('Candidate:N', sort=list(candidates)),
).properties(width=600, height=380, title='RevReCom: TV distance from p_n(d | Interior) to candidate distributions')

tv_chart.save('revrecom_candidate_tv.png', ppi=300)
tv_chart

In [ ]:
mean_interior_df = pd.DataFrame([
    {'n': n, 'mean_interior_degree': float((DEGREE_GRID * pmf).sum())}
    for n, pmf in sorted(interior_pmfs.items())
])

mean_interior_chart = alt.Chart(mean_interior_df).mark_line(point=True, size=2).encode(
    x=alt.X('n:Q', title='Grid size n'),
    y=alt.Y('mean_interior_degree:Q', title='Mean interior degree', scale=alt.Scale(zero=False)),
).properties(width=600, height=380, title='RevReCom: mean interior district degree vs grid size n')

mean_interior_chart.save('revrecom_mean_interior_degree.png', ppi=300)
mean_interior_chart

In [ ]:
panels = {
    'revrecom_normal_panel.png': (REFERENCE_NORMAL_LABEL, 'Discrete Normal'),
    'revrecom_binomial_panel.png': (REFERENCE_BINOMIAL_LABEL, 'Binomial'),
    'revrecom_poisson_voronoi_panel.png': (POISSON_VORONOI_LABEL, 'Poisson-Voronoi'),
}

for out_file, (label, short) in panels.items():
    chart = make_panel_chart(
        panel_df({label: candidates[label]}, PANEL_SIZES),
        ['Empirical p_n(d | I)', label],
        f'RevReCom: p_n(d | Interior) vs {short} candidate',
        PANEL_SIZES,
    )
    chart.save(out_file, ppi=300)
    display(chart)